# Shiv Nadar University Chennai
## CS3807 -- Deep Learning Laboratory
### Experiments 8, 9: Effect of Attention Type and Attention Position in a Pretrained ResNet50

**Student Name:** CY Pranav Menon  
**Roll Number:** 24011101014  
**Degree & Branch:** B.Tech Artificial Intelligence & Data Science (Semester V)  
**Academic Year:** 2026--27  
**GitHub Repository:** [PranavMenon07/Deep-Learning](https://github.com/PranavMenon07/Deep-Learning)

---

### Objectives
1. Understand how different attention mechanisms (Channel Attention / SE, Spatial Attention, CBAM, Self-Attention, ECA, Coordinate Attention) modify representations in a pretrained ResNet50.
2. Investigate how the position ($P_1, P_2, P_3, P_4$) and multi-stage insertion of attention blocks influence classification performance.
3. Perform controlled ablation studies on attention ordering and spatial kernel size ($k \in \{3, 5, 7\}$).
4. Evaluate quantitative trade-offs (Accuracy, Macro F1, Parameter counts, Inference latency).
5. Perform qualitative feature analysis using internal attention maps and post-hoc Grad-CAM.

In [ ]:
# Environment Setup & Core Imports
import os
import sys
import time
import json
import random
import tarfile
import urllib.request
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision import models, transforms
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# Reproducibility seed
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch Version: {torch.__version__} | Running on: {device}")

### 1. Dataset Preparation: TensorFlow Flowers (`tf_flowers`)
The dataset contains 3,670 natural RGB images belonging to 5 classes: `daisy`, `dandelion`, `roses`, `sunflowers`, `tulips`.
We download the official tarball and maintain an exact **70% : 15% : 15%** split for Train, Validation, and Test sets.

In [ ]:
# Download & Extract tf_flowers
data_dir = os.path.join(os.getcwd(), 'dataset')
os.makedirs(data_dir, exist_ok=True)
tgz_file = os.path.join(data_dir, 'flower_photos.tgz')
dataset_url = "https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz"

if not os.path.exists(tgz_file):
    print("Downloading TensorFlow Flowers dataset (218 MB)...")
    urllib.request.urlretrieve(dataset_url, tgz_file)
    print("Download complete!")

photos_dir = os.path.join(data_dir, 'flower_photos')
if not os.path.exists(photos_dir):
    print("Extracting archive...")
    with tarfile.open(tgz_file, 'r:gz') as tar:
        tar.extractall(data_dir)
    print("Extraction complete!")

classes = sorted([d for d in os.listdir(photos_dir) if os.path.isdir(os.path.join(photos_dir, d)) and not d.startswith('.')])
class_to_idx = {cls: i for i, cls in enumerate(classes)}
idx_to_class = {i: cls for i, cls in enumerate(classes)}
print(f"Detected {len(classes)} classes: {class_to_idx}")

In [ ]:
# Dataset Splitting (70% : 15% : 15%)
all_samples = []
for cls in classes:
    c_dir = os.path.join(photos_dir, cls)
    imgs = [os.path.join(c_dir, f) for f in os.listdir(c_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    for img_path in imgs:
        all_samples.append((img_path, class_to_idx[cls]))

random.shuffle(all_samples)
n_total = len(all_samples)
n_train = int(0.70 * n_total)
n_val = int(0.15 * n_total)

train_files = all_samples[:n_train]
val_files = all_samples[n_train:n_train + n_val]
test_files = all_samples[n_train + n_val:]
print(f"Total: {n_total} | Train: {len(train_files)} (70%) | Val: {len(val_files)} (15%) | Test: {len(test_files)} (15%)")

# Preprocessing Pipelines
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(degrees=15),
        transforms.ColorJitter(brightness=0.1, contrast=0.1),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ]),
    'test': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
}

class FlowersDataset(Dataset):
    def __init__(self, items, transform=None):
        self.items = items
        self.transform = transform
    def __len__(self):
        return len(self.items)
    def __getitem__(self, idx):
        path, label = self.items[idx]
        img = Image.open(path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, label, path

train_loader = DataLoader(FlowersDataset(train_files[:640], transform=data_transforms['train']), batch_size=32, shuffle=True)
val_loader = DataLoader(FlowersDataset(val_files[:160], transform=data_transforms['val']), batch_size=32, shuffle=False)
test_loader = DataLoader(FlowersDataset(test_files[:200], transform=data_transforms['test']), batch_size=32, shuffle=False)
print("DataLoaders initialized successfully.")

### 2. Attention Mechanisms Implementation
Here we define:
- **Channel Attention (SE style):** $M_c(F) = \sigma(W_2 \, 	ext{ReLU}(W_1 \, 	ext{GAP}(F)))$
- **Spatial Attention:** $M_s(F) = \sigma(	ext{Conv}_{k 	imes k}([	ext{Mean}_c(F) \,\|\, 	ext{Max}_c(F)]))$
- **CBAM Attention:** Sequential $F 	o M_c 	o M_s 	o F_{cs}$
- **Lightweight Self-Attention:** $A = 	ext{softmax}(Q K^T / \sqrt{d})$, $Y = A V$, $F_{SA} = F + \gamma \, 	ext{reshape}(Y)$
- **ECA-Net Module:** Fast 1D convolution over GAP without dimensionality reduction.
- **Coordinate Attention:** Directional 1D pooling along horizontal and vertical axes.

In [ ]:
# Attention Layer Modules
class ChannelAttention(nn.Module):
    def __init__(self, in_channels, reduction_ratio=16):
        super().__init__()
        reduced = max(1, in_channels // reduction_ratio)
        self.fc1 = nn.Linear(in_channels, reduced, bias=False)
        self.relu = nn.ReLU(inplace=True)
        self.fc2 = nn.Linear(reduced, in_channels, bias=False)
        self.sigmoid = nn.Sigmoid()
        self.last_weights = None

    def forward(self, x):
        b, c, h, w = x.size()
        z = x.view(b, c, -1).mean(dim=2)
        w_att = self.sigmoid(self.fc2(self.relu(self.fc1(z))))
        self.last_weights = w_att.detach().cpu()
        return x * w_att.view(b, c, 1, 1)

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        padding = (kernel_size - 1) // 2
        self.conv = nn.Conv2d(2, 1, kernel_size=kernel_size, padding=padding, bias=False)
        self.sigmoid = nn.Sigmoid()
        self.last_map = None

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        s_cat = torch.cat([avg_out, max_out], dim=1)
        s_att = self.sigmoid(self.conv(s_cat))
        self.last_map = s_att.detach().cpu()
        return x * s_att

class CBAMAttention(nn.Module):
    def __init__(self, in_channels, reduction_ratio=16, kernel_size=7, order='channel_first'):
        super().__init__()
        self.order = order
        self.ca = ChannelAttention(in_channels, reduction_ratio)
        self.sa = SpatialAttention(kernel_size)

    def forward(self, x):
        if self.order == 'channel_first':
            return self.sa(self.ca(x))
        else:
            return self.ca(self.sa(x))

class LightweightSelfAttention(nn.Module):
    def __init__(self, in_channels, reduction_ratio=8):
        super().__init__()
        self.inter_channels = max(1, in_channels // reduction_ratio)
        self.q_conv = nn.Conv2d(in_channels, self.inter_channels, 1, bias=False)
        self.k_conv = nn.Conv2d(in_channels, self.inter_channels, 1, bias=False)
        self.v_conv = nn.Conv2d(in_channels, in_channels, 1, bias=False)
        self.gamma = nn.Parameter(torch.zeros(1))
        self.last_attn_matrix = None

    def forward(self, x):
        b, c, h, w = x.size()
        n = h * w
        q = self.q_conv(x).view(b, self.inter_channels, n).permute(0, 2, 1)
        k = self.k_conv(x).view(b, self.inter_channels, n)
        energy = torch.bmm(q, k) / (self.inter_channels ** 0.5)
        attn = F.softmax(energy, dim=-1)
        self.last_attn_matrix = attn.detach().cpu()
        v = self.v_conv(x).view(b, c, n)
        out = torch.bmm(v, attn.permute(0, 2, 1)).view(b, c, h, w)
        return x + self.gamma * out

class ECAModule(nn.Module):
    def __init__(self, in_channels, gamma=2, b=1):
        super().__init__()
        k = int(abs((np.log2(in_channels) / gamma) + (b / gamma)))
        kernel_size = k if k % 2 == 1 else k + 1
        self.conv = nn.Conv1d(1, 1, kernel_size=kernel_size, padding=(kernel_size - 1) // 2, bias=False)
        self.sigmoid = nn.Sigmoid()
        self.last_weights = None

    def forward(self, x):
        b, c, h, w = x.size()
        z = x.view(b, c, -1).mean(dim=2)
        y = self.conv(z.unsqueeze(1))
        att = self.sigmoid(y).squeeze(1)
        self.last_weights = att.detach().cpu()
        return x * att.view(b, c, 1, 1)

class CoordinateAttention(nn.Module):
    def __init__(self, in_channels, reduction=16):
        super().__init__()
        self.pool_h = nn.AdaptiveAvgPool2d((None, 1))
        self.pool_w = nn.AdaptiveAvgPool2d((1, None))
        mip = max(8, in_channels // reduction)
        self.conv1 = nn.Conv2d(in_channels, mip, 1)
        self.bn1 = nn.BatchNorm2d(mip)
        self.act = nn.ReLU(inplace=True)
        self.conv_h = nn.Conv2d(mip, in_channels, 1)
        self.conv_w = nn.Conv2d(mip, in_channels, 1)

    def forward(self, x):
        b, c, h, w = x.size()
        x_h = self.pool_h(x)
        x_w = self.pool_w(x).permute(0, 1, 3, 2)
        y = self.act(self.bn1(self.conv1(torch.cat([x_h, x_w], dim=2))))
        x_h, x_w = torch.split(y, [h, w], dim=2)
        a_h = torch.sigmoid(self.conv_h(x_h))
        a_w = torch.sigmoid(self.conv_w(x_w.permute(0, 1, 3, 2)))
        return x * a_h * a_w

print("All attention modules defined.")

In [ ]:
# ResNet50 Backbone with Modular Attention Insertion Points (P1, P2, P3, P4)
class ResNet50WithAttention(nn.Module):
    def __init__(self, attention_type=None, position='P3', num_classes=5, dropout_rate=0.5,
                 spatial_kernel=7, attention_order='channel_first', multiple_positions=None):
        super().__init__()
        weights = models.ResNet50_Weights.DEFAULT
        resnet = models.resnet50(weights=weights)

        self.stem = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu, resnet.maxpool)
        self.layer1 = resnet.layer1  # P1: 256 ch, 56x56
        self.layer2 = resnet.layer2  # P2: 512 ch, 28x28
        self.layer3 = resnet.layer3  # P3: 1024 ch, 14x14
        self.layer4 = resnet.layer4  # P4: 2048 ch, 7x7

        stage_channels = {'P1': 256, 'P2': 512, 'P3': 1024, 'P4': 2048}

        def build_att(att_type, ch):
            if att_type == 'channel':
                return ChannelAttention(ch, reduction_ratio=16)
            elif att_type == 'spatial':
                return SpatialAttention(kernel_size=spatial_kernel)
            elif att_type in ('cbam', 'channel_spatial'):
                return CBAMAttention(ch, reduction_ratio=16, kernel_size=spatial_kernel, order=attention_order)
            elif att_type == 'self_attention':
                return LightweightSelfAttention(ch)
            elif att_type == 'combined':
                return nn.Sequential(
                    CBAMAttention(ch, reduction_ratio=16, kernel_size=spatial_kernel),
                    LightweightSelfAttention(ch)
                )
            elif att_type == 'eca':
                return ECAModule(ch)
            elif att_type == 'coordinate':
                return CoordinateAttention(ch)
            return nn.Identity()

        self.att_P1 = nn.Identity()
        self.att_P2 = nn.Identity()
        self.att_P3 = nn.Identity()
        self.att_P4 = nn.Identity()

        if multiple_positions:
            for p in multiple_positions:
                setattr(self, f"att_{p}", build_att(attention_type, stage_channels[p]))
        elif attention_type and attention_type != 'none':
            setattr(self, f"att_{position}", build_att(attention_type, stage_channels[position]))

        self.gap = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout = nn.Dropout(p=dropout_rate)
        self.classifier = nn.Linear(2048, num_classes)

    def forward(self, x):
        x = self.stem(x)
        x = self.att_P1(self.layer1(x))
        x = self.att_P2(self.layer2(x))
        x = self.att_P3(self.layer3(x))
        x = self.att_P4(self.layer4(x))
        x = self.gap(x)
        x = self.dropout(torch.flatten(x, 1))
        return self.classifier(x)

print("ResNet50 with Attention instantiated successfully.")

In [ ]:
# Training and Evaluation Procedures
def train_model(model, train_loader, val_loader, epochs=4, lr=1e-3):
    criterion = nn.CrossEntropyLoss()
    # Freeze stem, layer1, layer2 for transfer learning warm-up
    for p in model.stem.parameters(): p.requires_grad = False
    for p in model.layer1.parameters(): p.requires_grad = False
    for p in model.layer2.parameters(): p.requires_grad = False
    for p in model.layer3.parameters(): p.requires_grad = False
    for p in model.layer4.parameters(): p.requires_grad = False

    optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr, weight_decay=1e-4)
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

    start = time.time()
    for ep in range(epochs):
        model.train()
        r_loss, correct, total = 0.0, 0, 0
        for imgs, lbls, _ in train_loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            optimizer.zero_grad()
            out = model(imgs)
            loss = criterion(out, lbls)
            loss.backward()
            optimizer.step()
            r_loss += loss.item() * imgs.size(0)
            preds = out.argmax(dim=1)
            correct += (preds == lbls).sum().item()
            total += lbls.size(0)
        
        train_acc = correct / total
        train_loss = r_loss / total

        model.eval()
        v_loss, v_corr, v_tot = 0.0, 0, 0
        with torch.no_grad():
            for imgs, lbls, _ in val_loader:
                imgs, lbls = imgs.to(device), lbls.to(device)
                out = model(imgs)
                v_loss += criterion(out, lbls).item() * imgs.size(0)
                v_corr += (out.argmax(dim=1) == lbls).sum().item()
                v_tot += lbls.size(0)

        history['train_loss'].append(train_loss)
        history['val_loss'].append(v_loss / v_tot)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(v_corr / v_tot)
        print(f"Epoch {ep+1}/{epochs}: Train Acc={train_acc:.4f} | Val Acc={v_corr/v_tot:.4f}")

    return history, time.time() - start

def evaluate_model(model, loader):
    model.eval()
    preds, targets = [], []
    start = time.time()
    with torch.no_grad():
        for imgs, lbls, _ in loader:
            imgs = imgs.to(device)
            out = model(imgs)
            preds.extend(out.argmax(dim=1).cpu().numpy())
            targets.extend(lbls.numpy())
    inf_time_ms = ((time.time() - start) / len(targets)) * 1000.0

    acc = accuracy_score(targets, preds)
    prec = precision_score(targets, preds, average='macro', zero_division=0)
    rec = recall_score(targets, preds, average='macro', zero_division=0)
    f1 = f1_score(targets, preds, average='macro', zero_division=0)
    cm = confusion_matrix(targets, preds)
    tot_params = sum(p.numel() for p in model.parameters())

    return {
        'accuracy': float(acc), 'precision': float(prec),
        'recall': float(rec), 'f1': float(f1),
        'total_params': int(tot_params), 'inf_time_ms': float(inf_time_ms),
        'confusion_matrix': cm
    }

### 3. Experimental Ablation Results
Below is the summary of quantitative metrics recorded across all attention variants, insertion points, and kernel sizes.

In [ ]:
# Load saved experimental results
with open('experiment_results.json', 'r') as f:
    res = json.load(f)

print(f"{'Model Configuration':<25} | {'Position':<8} | {'Accuracy':<9} | {'Precision':<9} | {'Recall':<8} | {'Macro F1':<9} | {'Parameters':<11} | {'Inf. Time':<10}")
print("-" * 105)
for k, v in res.items():
    pos = 'P3' if 'P3' in k else ('P2' if 'P2' in k else ('P4' if 'P4' in k else '--'))
    print(f"{k:<25} | {pos:<8} | {v['accuracy']*100:6.2f}%   | {v['precision']*100:6.2f}%   | {v['recall']*100:6.2f}%  | {v['f1_score']*100:6.2f}%   | {v['total_params']:<11,d} | {v['inference_time_ms']:5.2f} ms")

### 4. Visualizations & Interpretability
All 16 scientific figures generated during the experiments are displayed below.

In [ ]:
# Display All Key Figures
import matplotlib.image as mpimg

plot_files = [
    ("Plot 1: Dataset Samples", "plots/plot01_sample_dataset_images.png"),
    ("Plot 2: Accuracy Curves", "plots/plot02_train_val_accuracy_curves.png"),
    ("Plot 3: Loss Curves", "plots/plot03_train_val_loss_curves.png"),
    ("Plot 4: Confusion Matrices", "plots/plot04_confusion_matrices.png"),
    ("Plot 5: Feature Maps Before/After", "plots/plot05_feature_maps_before_after.png"),
    ("Plot 6: Channel Attention Weights", "plots/plot06_channel_attention_weights.png"),
    ("Plot 7: Spatial Attention Overlay", "plots/plot07_spatial_attention_heatmap_overlay.png"),
    ("Plot 8: Self-Attention Response", "plots/plot08_self_attention_response.png"),
    ("Plot 9: Grad-CAM Comparison", "plots/plot09_gradcam_baseline_vs_attention.png"),
    ("Plot 10: Multi-Stage Spatial Progression", "plots/plot10_attention_maps_by_position.png"),
    ("Plot 11: Accuracy vs Parameters", "plots/plot11_accuracy_vs_parameters.png"),
    ("Plot 12: Macro F1 vs Inference Latency", "plots/plot12_f1_vs_inference_time.png"),
    ("Plot 13: Attention Type Ablation", "plots/plot13_attention_type_comparison.png"),
    ("Plot 14: Attention Position Ablation", "plots/plot14_attention_position_comparison.png"),
    ("Plot 15: Spatial Kernel Ablation", "plots/plot15_spatial_kernel_ablation.png"),
    ("Plot 16: Multi-Image Qualitative Comparison", "plots/plot16_qualitative_multi_image_comparison.png")
]

for title, path in plot_files:
    if os.path.exists(path):
        plt.figure(figsize=(12, 6))
        img = mpimg.imread(path)
        plt.imshow(img)
        plt.title(title, fontsize=14, fontweight='bold')
        plt.axis('off')
        plt.show()

### 5. Final Key Takeaways
1. **Channel Attention (SE style)** at stage $P_3$ attained the highest classification accuracy (**91.00%**) and Macro F1 (**89.54%**) by selecting category-specific flower color and texture filters.
2. **Topological Position Matters:** Inserting attention at deeper stages ($P_3, P_4$) outperforms shallower stages ($P_2$), confirming that attention operates most effectively on semantically mature convolutional representations.
3. **ECA-Net Pareto Efficiency:** Efficient Channel Attention matches competitive performance while introducing only 5 additional parameters over the 23.5M parameter backbone.
4. **Interpretability Agreement:** Both internal attention maps and post-hoc Grad-CAM visualizations demonstrate that attention mechanisms effectively suppress irrelevant background clutter (soil, foliage) and concentrate on discriminative petal morphology.